# fMRI · 06 · Adapter checkpoint sweep

**Why:** the token-adapter training loss (single-timestep diffusion MSE) is a noisy, unreliable proxy for generation quality after full sampling (50 steps + CFG). This notebook generates with several adapter checkpoints and scores each by CLIP similarity to the real image — the same metric as Experiment 5 — so you pick the checkpoint by what matters.

In [ ]:
# Run from the PROJECT ROOT so relative paths (configs/, data/, outputs/)
# resolve exactly like the scripts do.
import sys, os
_root = os.getcwd()
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, 'src')) and os.path.isdir(os.path.join(_root, 'configs')):
        break
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
print('project root:', os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from src.utils import load_config, get_experiment_paths
from src.generation import discover_adapter_checkpoints, sweep_adapter_checkpoints
cfg = load_config('configs/fMRI/exp04_generation.yaml')
# Force a mode that loads the adapter (adapter | adapter_lowlevel):
cfg['generation']['mode'] = 'adapter'
paths = get_experiment_paths(cfg, ensure=False)
checkpoints = discover_adapter_checkpoints(paths.checkpoints)
print(f'{len(checkpoints)} checkpoints in {paths.checkpoints}:')
for label, p in checkpoints: print(' -', label, '->', p.name)

## Run the sweep
Small `num_samples`/`num_inference_steps` for a quick pass; raise both for the numbers that go in the report.

In [ ]:
decoder_checkpoint = cfg.get('generation.decoder_checkpoint') or 'outputs/exp03_lowlevel_multitask/checkpoints/best.pt'
result = sweep_adapter_checkpoints(cfg, decoder_checkpoint, checkpoints=checkpoints,
                                   conditions=['correct', 'permuted'],
                                   num_samples=6, num_inference_steps=50)
print('saved to:', result['out_dir'])

## Summary (one row per checkpoint × condition)

In [ ]:
result['summary'].pivot_table(index=['checkpoint','epoch'], columns='condition', values='mean_clip_similarity')

## Margin correct − best control, per checkpoint
`uses_fmri_signal=True` when the correct condition clearly beats the controls with that specific checkpoint (same criterion as Experiment 2, applied to generation).

In [ ]:
result['margins']

## Quality vs training epoch

In [ ]:
%matplotlib inline

In [ ]:
from PIL import Image as _Image
plt.figure(figsize=(9, 5)); plt.imshow(_Image.open(f"{result['out_dir']}/checkpoint_sweep_quality.png")); plt.axis('off'); plt.show()

## Pick the winner
Highest `margin` (correct − best control), tie-broken by `mean_clip_similarity` of `correct` — that is the checkpoint to use in `generation.adapter_checkpoint`, not necessarily `adapter_best.pt`.

In [ ]:
best_row = result['margins'].sort_values('margin', ascending=False).iloc[0]
print('Recommended checkpoint:', best_row['checkpoint'], '| epoch:', best_row['epoch'], '| margin:', round(best_row['margin'], 4))